# 04 — Regional Robustness (RQ3)

**Question.** Does the global pattern (events rising faster than fatalities; most of the lethality decline driven by within-type lethality reductions rather than mix shifts) hold inside every ACLED region, or is it driven by one or two regions?

**Method.** Repeat RQ1 (annual events vs fatalities, correlation, log-slope comparison) and RQ2 (shift-share decomposition of per-event lethality) separately for each region. For the decomposition we use each region's first and last available year inside the 2014–2025 window as its own base/end periods — this makes US-and-Canada (starts 2019) comparable on its own terms, with a clear caveat.

**Takeaway.** Reported at the bottom.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

PROJECT = Path('/Users/zhangtingmin/Desktop/PhD/Year_1_Sem_2/POLI_3148/Assignment 1 copy 5')
DERIVED = PROJECT / 'data' / 'derived'
FIGDIR = PROJECT / 'docs' / 'figures'
df = pd.read_parquet(DERIVED / 'acled_clean.parquet')
REGIONS = ['Africa','Asia-Pacific','Europe-Central-Asia','Latin-America-Caribbean','Middle-East','US-and-Canada']
print('Loaded:', df.shape, '| Regions:', df['REGION'].nunique())

Loaded: (838798, 15) | Regions: 6


## Small multiples: annual events vs. fatalities, one panel per region

In [ ]:
import subprocess, sys
# The events-vs-fatalities small-multiples figure is generated by
# code/build_regional_small_multiples.py with uniform horizontal x-axis
# labels, teal/red palette, and a locked 2014-2025 range. Delegating here
# keeps the notebook and the published report in sync.
subprocess.run([sys.executable, str(PROJECT / 'code' / 'build_regional_small_multiples.py')], check=True)
print('Rebuilt:', FIGDIR / '04_regional_small_multiples.html')

## Per-region RQ1/RQ2 statistics

For each region: Pearson r, log-slopes for events and fatalities, and shift-share decomposition on `EVENT_TYPE`.

In [3]:
def shift_share(df_long, year0, year1, group_col):
    d0 = df_long[df_long['YEAR']==year0].set_index(group_col)
    d1 = df_long[df_long['YEAR']==year1].set_index(group_col)
    keys = sorted(set(d0.index) | set(d1.index))
    d0 = d0.reindex(keys).fillna(0); d1 = d1.reindex(keys).fillna(0)
    s0 = d0['EVENTS']/max(d0['EVENTS'].sum(),1)
    s1 = d1['EVENTS']/max(d1['EVENTS'].sum(),1)
    l0 = np.where(d0['EVENTS']>0, d0['FATALITIES']/d0['EVENTS'].replace(0,np.nan), 0)
    l1 = np.where(d1['EVENTS']>0, d1['FATALITIES']/d1['EVENTS'].replace(0,np.nan), 0)
    L0 = float((s0*l0).sum()); L1 = float((s1*l1).sum())
    comp   = float(((s1 - s0) * l0).sum())
    within = float((s0 * (l1 - l0)).sum())
    inter  = float(((s1 - s0) * (l1 - l0)).sum())
    return L0, L1, comp, within, inter

rows = []
for reg in REGIONS:
    sub = df[df['REGION']==reg]
    ann = sub.groupby('YEAR')[['EVENTS','FATALITIES']].sum().reset_index()
    ann = ann[(ann['EVENTS']>0) & (ann['FATALITIES']>0)]
    y0, y1 = int(ann['YEAR'].min()), int(ann['YEAR'].max())
    x = ann['YEAR'].to_numpy()
    ev = stats.linregress(x, np.log(ann['EVENTS'].to_numpy()))
    ft = stats.linregress(x, np.log(ann['FATALITIES'].to_numpy()))
    pear = stats.pearsonr(ann['EVENTS'], ann['FATALITIES'])
    by_type = sub.groupby(['YEAR','EVENT_TYPE'])[['EVENTS','FATALITIES']].sum().reset_index()
    L0,L1,c,w,i = shift_share(by_type, y0, y1, 'EVENT_TYPE')
    dL = L1 - L0
    rows.append({
        'REGION': reg, 'years': f'{y0}–{y1}',
        'pearson_r': round(pear.statistic, 3),
        'ev_slope_%yr': round((np.exp(ev.slope)-1)*100, 1),
        'ft_slope_%yr': round((np.exp(ft.slope)-1)*100, 1),
        'slope_diff(ft-ev)_pp': round((np.exp(ft.slope)-np.exp(ev.slope))*100, 1),
        'L0': round(L0,2), 'L1': round(L1,2), 'dL': round(dL,2),
        'comp_share': round(c/dL*100,0) if dL!=0 else np.nan,
        'within_share': round(w/dL*100,0) if dL!=0 else np.nan,
        'inter_share': round(i/dL*100,0) if dL!=0 else np.nan,
    })
region_tbl = pd.DataFrame(rows)
region_tbl

,REGION,years,pearson_r,ev_slope_%yr,ft_slope_%yr,slope_diff(ft-ev)_pp,L0,L1,dL,comp_share,within_share,inter_share
0,Africa,2014–2025,0.931,13.6,8.4,-5.2,2.44,1.42,-1.02,25.0,81.0,-5.0
1,Asia-Pacific,2014–2025,0.643,21.2,12.2,-8.9,0.83,0.31,-0.52,66.0,57.0,-22.0
2,Europe-Central-Asia,2017–2025,0.911,66.1,182.9,116.8,0.01,0.59,0.58,0.0,229.0,-129.0
3,Latin-America-Caribbean,2017–2025,0.959,37.7,34.8,-2.9,0.72,0.54,-0.18,102.0,-4.0,2.0
4,Middle-East,2014–2025,0.352,50.5,26.4,-24.1,1.80,0.46,-1.34,78.0,62.0,-39.0
5,US-and-Canada,2020–2025,0.017,-3.1,-21.0,-17.9,0.00,0.00,-0.00,75.0,-24.0,50.0


In [4]:
region_tbl.to_csv(DERIVED / 'regional_summary.csv', index=False)
print('Saved:', DERIVED / 'regional_summary.csv')

Saved: /Users/zhangtingmin/Desktop/PhD/Year_1_Sem_2/POLI_3148/Assignment 1 copy 5/data/derived/regional_summary.csv


## Per-region lethality trajectory

In [ ]:
import subprocess, sys
# The lethality-by-region figure is generated by code/build_regional_lethality.py,
# which renders a 2x3 small-multiples layout with per-panel linear axes, a muted
# global reference line, endpoint annotations, and horizontal year labels.
# Delegating here keeps the notebook and the published report in sync.
subprocess.run([sys.executable, str(PROJECT / 'code' / 'build_regional_lethality.py')], check=True)
print('Rebuilt:', FIGDIR / '04_regional_lethality.html')

## Per-region decomposition bar chart

In [6]:
fig_dec = go.Figure()
fig_dec.add_trace(go.Bar(x=region_tbl['REGION'], y=region_tbl['comp_share'],   name='Composition %', marker_color='steelblue'))
fig_dec.add_trace(go.Bar(x=region_tbl['REGION'], y=region_tbl['within_share'], name='Within-type %', marker_color='firebrick'))
fig_dec.add_trace(go.Bar(x=region_tbl['REGION'], y=region_tbl['inter_share'],  name='Interaction %', marker_color='grey'))
fig_dec.update_layout(title='Share of ΔL explained by composition vs within-type, by region',
                      barmode='stack', yaxis_title='% of ΔL', height=450, xaxis_tickangle=-20)
fig_dec.write_html(FIGDIR / '04_regional_decomposition.html', include_plotlyjs='cdn')
fig_dec.show()

## Takeaway

Results saved to `data/derived/regional_summary.csv`. Narrative interpretation of which regions conform to / deviate from the global pattern is deferred to the report.